In [1]:
import pandas as pd
import numpy as np

# steps 

# 01 loading the dataset

In [2]:
df = pd.read_csv("powerplant_data.csv")
df.head()

,AT,V,AP,RH,PE
0,8.34,40.77,1010.84,90.01,480.48
1,23.64,58.49,1011.40,74.20,445.75
2,29.74,56.90,1007.15,41.91,438.76
3,19.07,49.69,1007.22,76.79,453.09
4,11.80,40.66,1017.13,97.20,464.43


# 02 splitingt 

In [3]:
X = df.drop("PE", axis=1)
y = df["PE"]

# 03 train test split 

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state=42
)

# 04 scaling 

In [5]:
from sklearn.preprocessing import StandardScaler
scale = StandardScaler()
X_train_sc = scale.fit_transform(X_train)
X_test_sc = scale.transform(X_test)

# 05 converting the dataset to Tensors

In [6]:
import torch 
import  torch.nn as nn 

In [7]:
# these all values are stored in the RAM of our laptop or i am using mac and it store's in my RAM

X_train_tensor = torch.tensor(X_train_sc, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1) # here -1 will take all columns value and 1 is like [[]] it make like this 

X_test_tensor = torch.tensor(X_test_sc, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1) # here -1 will take all columns value and 1 is like [[]] it make like this 


# 06 making TensordataSet and DataLoader 

In [8]:
from torch.utils.data import DataLoader, TensorDataset
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32 ,shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

# 07 ANN MODEL 

In [9]:
# Define our ANN Model

class ANN(nn.Module):
    def __init__(self):
        super(ANN, self).__init__()

        self.model = nn.Sequential(

            # 1st hidden layer
            nn.Linear(X_train.shape[1], 6), # input_feature, output_feature
            nn.ReLU(),

            # 2nd hidden layer
            nn.Linear(6, 6), # input_feature, output_feature
            nn.ReLU(),

            # Output layer
            nn.Linear(6, 1),
        )

    def forward(self, x):
        return self.model(x)

In [12]:
import torch.optim as optim

model = ANN()

# loss, optimizer
crietrion = nn.MSELoss()
optimizer = optim.Adam(model.parameters())

In [13]:
# Train the ANN
train_losses = []
val_losses = []

best_val_loss = float("inf")

epochs = 100

for epoch in range(epochs):
    model.train()
    running_loss = 0.0 # tot training loss for 1 epoch
    
    for xb, yb in train_loader:
        # xb = features of 1 batch
        # yb = labels of 1 batch
        optimizer.zero_grad()
        
        outputs = model(xb) # forward prop....predicted outputs for this batch
        loss = crietrion(outputs, yb) # compute loss
        loss.backward() # back prop.. compute gradients
        optimizer.step() # params update
        
        running_loss += loss.item() # loss is a tensor => py float

    epoch_train_loss = running_loss / len(train_loader)
    train_losses.append(epoch_train_loss)


    # Validation
    model.eval()
    running_val_loss = 0.0

    with torch.no_grad(): # no gradients compute
        for xb, yb in test_loader:
            outputs = model(xb)
            loss = crietrion(outputs, yb)
            running_val_loss += loss

    epoch_val_loss = running_val_loss / len(test_loader)
    val_losses.append(epoch_val_loss)

    print(f"epoch {epoch+1}/{epochs} ==> train loss = {epoch_train_loss} & val loss = {epoch_val_loss}")

    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), "best_model.pt") #.pt or .pth

epoch 1/100 ==> train loss = 206194.97708333333 & val loss = 204648.1875
epoch 2/100 ==> train loss = 199695.76940104167 & val loss = 190566.09375
epoch 3/100 ==> train loss = 173065.77278645834 & val loss = 150789.21875
epoch 4/100 ==> train loss = 122029.82415364584 & val loss = 92466.5234375
epoch 5/100 ==> train loss = 66889.68375651042 & val loss = 46021.2109375
epoch 6/100 ==> train loss = 33831.59735514323 & val loss = 25907.994140625
epoch 7/100 ==> train loss = 22076.012540690102 & val loss = 19577.021484375
epoch 8/100 ==> train loss = 17704.42207845052 & val loss = 16152.8076171875
epoch 9/100 ==> train loss = 14630.225406901041 & val loss = 13129.9677734375
epoch 10/100 ==> train loss = 11748.617704264323 & val loss = 10255.560546875
epoch 11/100 ==> train loss = 8975.866131591796 & val loss = 7598.9990234375
epoch 12/100 ==> train loss = 6452.082706705729 & val loss = 5241.30712890625
epoch 13/100 ==> train loss = 4302.313964335123 & val loss = 3349.61328125
epoch 14/100 =